In [ ]:
import os
import re
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import wfdb

from scipy.signal import butter, filtfilt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import tensorflow as tf
from tensorflow.keras import layers, Model

warnings.filterwarnings("ignore")

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)

In [ ]:
print("TensorFlow GPUs:")
print(tf.config.list_physical_devices("GPU"))

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(".")

ECG_DIR = PROJECT_ROOT / "DataSets" / "WFDB_Ningbo"

print("ECG directory:")
print(ECG_DIR.resolve())

print("\nExists:", ECG_DIR.exists())

In [ ]:
print(list(Path("DataSets").iterdir()))

In [ ]:
items = list(ECG_DIR.iterdir())

print("Number of items:", len(items))

for item in items[:20]:
    print(item)

In [ ]:
hea_files = list(ECG_DIR.rglob("*.hea"))

print("Total .hea files found:", len(hea_files))

print("\nFirst 10:")
for file in hea_files[:10]:
    print(file)

In [ ]:
mat_files = list(ECG_DIR.rglob("*.mat"))

print("HEA files:", len(hea_files))
print("MAT files:", len(mat_files))

In [ ]:
first_record = hea_files[0].with_suffix("")

print("Record path:")
print(first_record)

In [ ]:
record = wfdb.rdrecord(str(first_record))

print("Signal shape:", record.p_signal.shape)
print("Sampling frequency:", record.fs)
print("Number of leads:", record.n_sig)
print("Lead names:", record.sig_name)

In [ ]:
print("Comments:")

for comment in record.comments:
    print(comment)

In [ ]:
signal = record.p_signal

plt.figure(figsize=(16, 12))

for i in range(record.n_sig):

    plt.subplot(6, 2, i + 1)

    plt.plot(signal[:, i])

    plt.title(record.sig_name[i])
    plt.xlabel("Samples")
    plt.ylabel("Amplitude")

plt.tight_layout()
plt.show()

In [ ]:
fs = record.fs

seconds = 5

samples = int(seconds * fs)

plt.figure(figsize=(16, 12))

for i in range(12):

    plt.subplot(6, 2, i + 1)

    plt.plot(
        np.arange(samples) / fs,
        signal[:samples, i]
    )

    plt.title(record.sig_name[i])
    plt.xlabel("Time (seconds)")
    plt.ylabel("Amplitude")

plt.tight_layout()
plt.show()

In [ ]:
def extract_metadata(record_path):

    record = wfdb.rdrecord(str(record_path))

    age = None
    sex = None
    diagnosis = None

    for comment in record.comments:

        comment = comment.strip()

        if comment.startswith("Age:"):
            age = comment.split(":", 1)[1].strip()

        elif comment.startswith("Sex:"):
            sex = comment.split(":", 1)[1].strip()

        elif comment.startswith("Dx:"):
            diagnosis = comment.split(":", 1)[1].strip()

    return {
        "age": age,
        "sex": sex,
        "diagnosis": diagnosis,
        "sampling_frequency": record.fs,
        "num_leads": record.n_sig,
        "num_samples": record.p_signal.shape[0],
        "lead_names": ",".join(record.sig_name)
    }

In [ ]:
metadata = extract_metadata(first_record)

metadata

In [ ]:
def extract_diagnosis_codes(record_path):

    record = wfdb.rdrecord(str(record_path))

    diagnosis = None

    for comment in record.comments:

        comment = comment.strip()

        if comment.startswith("Dx:"):
            diagnosis = comment.split(":", 1)[1].strip()
            break

    if diagnosis is None:
        return []

    codes = [
        code.strip()
        for code in diagnosis.split(",")
        if code.strip()
    ]

    return codes

In [ ]:
codes = extract_diagnosis_codes(first_record)

print("Diagnosis codes:", codes)

In [ ]:
NORMAL_CODE = "426783006"


def create_binary_label(codes):

    if len(codes) == 0:
        return None

    # Normal only
    if len(codes) == 1 and codes[0] == NORMAL_CODE:
        return 0

    # Anything else = abnormal
    return 1

In [ ]:
print(create_binary_label(["426783006"]))
print(create_binary_label(["164889003"]))

In [ ]:
records = [
    file.with_suffix("")
    for file in hea_files
]

print("Total records:", len(records))

In [ ]:
metadata_list = []

for i, record_path in enumerate(records):

    try:

        info = extract_metadata(record_path)

        codes = extract_diagnosis_codes(record_path)

        label = create_binary_label(codes)

        metadata_list.append({
            "record_path": str(record_path),
            "age": info["age"],
            "sex": info["sex"],
            "diagnosis_codes": ",".join(codes),
            "label": label,
            "sampling_frequency": info["sampling_frequency"],
            "num_leads": info["num_leads"],
            "num_samples": info["num_samples"],
            "lead_names": info["lead_names"]
        })

    except Exception as e:

        print(
            f"Error processing {record_path}: {e}"
        )

    if (i + 1) % 1000 == 0:
        print(f"Processed {i + 1}/{len(records)}")

In [ ]:
ecg_df = pd.DataFrame(metadata_list)

print("Shape:", ecg_df.shape)

ecg_df.head()

In [ ]:
metadata_path = PROJECT_ROOT / "DataSets" / "ecg_metadata.csv"

ecg_df.to_csv(
    metadata_path,
    index=False
)

print("Saved:")
print(metadata_path.resolve())

In [ ]:
ecg_df.isnull().sum()

In [ ]:
print("Total records:", len(ecg_df))

print(
    "Records with diagnosis:",
    ecg_df["label"].notna().sum()
)

In [ ]:
ecg_df = ecg_df[
    (ecg_df["num_leads"] == 12) &
    (ecg_df["sampling_frequency"] == 500) &
    (ecg_df["label"].notna())
].copy()

ecg_df.reset_index(
    drop=True,
    inplace=True
)

print("Valid ECG records:", len(ecg_df))

In [ ]:
print(
    ecg_df["label"].value_counts()
)

In [ ]:
plt.figure(figsize=(6, 4))

sns.countplot(
    data=ecg_df,
    x="label"
)

plt.title("Normal vs Abnormal ECG")
plt.xlabel("Label")
plt.ylabel("Number of ECGs")

plt.show()

In [ ]:
class_percentage = (
    ecg_df["label"]
    .value_counts(normalize=True)
    .mul(100)
)

print(class_percentage)

In [ ]:
def bandpass_filter(
    signal,
    lowcut=0.5,
    highcut=40.0,
    fs=500,
    order=3
):

    nyquist = fs / 2

    low = lowcut / nyquist
    high = highcut / nyquist

    b, a = butter(
        order,
        [low, high],
        btype="band"
    )

    filtered_signal = filtfilt(
        b,
        a,
        signal,
        axis=0
    )

    return filtered_signal

In [ ]:
def normalize_ecg(signal):

    mean = np.mean(
        signal,
        axis=0,
        keepdims=True
    )

    std = np.std(
        signal,
        axis=0,
        keepdims=True
    )

    normalized = (
        signal - mean
    ) / (
        std + 1e-8
    )

    return normalized

In [ ]:
def preprocess_ecg(record_path):

    record = wfdb.rdrecord(
        str(record_path)
    )

    signal = record.p_signal.astype(
        np.float32
    )

    # Make sure it is 12 lead
    if signal.shape[1] != 12:
        raise ValueError(
            f"Expected 12 leads, got {signal.shape[1]}"
        )

    # Filtering
    signal = bandpass_filter(
        signal,
        lowcut=0.5,
        highcut=40,
        fs=record.fs
    )

    # Normalization
    signal = normalize_ecg(signal)

    return signal.astype(np.float32)

In [ ]:
processed_signal = preprocess_ecg(
    first_record
)

print(
    "Processed shape:",
    processed_signal.shape
)

print(
    "Mean:",
    processed_signal.mean()
)

print(
    "Std:",
    processed_signal.std()
)

In [ ]:
lead = 1

plt.figure(figsize=(15, 5))

plt.plot(
    signal[:2500, lead],
    label="Raw ECG"
)

plt.plot(
    processed_signal[:2500, lead],
    label="Processed ECG"
)

plt.title(
    f"Lead {record.sig_name[lead]} - Raw vs Processed"
)

plt.xlabel("Samples")
plt.ylabel("Amplitude")

plt.legend()

plt.show()

In [ ]:
train_df, temp_df = train_test_split(
    ecg_df,
    test_size=0.30,
    stratify=ecg_df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
print("TRAIN")
print(train_df["label"].value_counts(normalize=True))

print("\nVALIDATION")
print(val_df["label"].value_counts(normalize=True))

print("\nTEST")
print(test_df["label"].value_counts(normalize=True))